# Lab 6: Iceberg & MinIO with Spark

Notebook này minh họa cách đọc dữ liệu từ file CSV, sau đó lưu thành định dạng Iceberg table trên MinIO.

In [ ]:
from pyspark.sql import SparkSession
import os

# Khởi tạo SparkSession với cấu hình Iceberg và MinIO
spark = SparkSession.builder \
    .appName("Iceberg-MinIO-Demo") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.demo.type", "rest") \
    .config("spark.sql.catalog.demo.uri", "http://rest:8181") \
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.demo.warehouse", "s3a://warehouse/") \
    .config("spark.sql.defaultCatalog", "demo") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session initiated with Iceberg REST Catalog")

### Đọc dữ liệu CSV từ MinIO
File `nyc_taxi_data.csv` đã được docker-compose (qua mc container) copy vào bucket `rawdata`.

In [ ]:
csv_path = "s3a://rawdata/csv/nyc_taxi_data.csv"

# Đọc CSV với header, Spark tự động infer schema
df_csv = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path)

df_csv.printSchema()
df_csv.show(5)

### Ghi dữ liệu xuống định dạng Iceberg
Tạo table `demo.nyc.taxis` và lưu toàn bộ data từ dataframe.

In [ ]:
# Tạo namespace nyc nếu chưa có
spark.sql("CREATE NAMESPACE IF NOT EXISTS nyc")

# Ghi đè vào bảng iceberg (sẽ tự động tạo bảng nếu chưa tồn tại)
print("Đang ghi dữ liệu vào bảng Iceberg demo.nyc.taxis...")
df_csv.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("demo.nyc.taxis")
print("Ghi dữ liệu thành công!")

### Xác nhận dữ liệu trong bảng Iceberg

In [ ]:
count_df = spark.sql("SELECT COUNT(*) AS cnt FROM demo.nyc.taxis")
print(f"Total Rows for NYC Taxi Data in Iceberg: {count_df.first().cnt}")

### Nâng cao: Schema Evolution (Thêm Cột Mới)
Thử thêm cột `fare_per_distance` và cập nhật dữ liệu. Iceberg hỗ trợ Schema Evolution mà không cần ghi lại toàn bộ bảng.

In [ ]:
spark.sql("ALTER TABLE demo.nyc.taxis ADD COLUMN fare_per_distance FLOAT")
spark.sql("UPDATE demo.nyc.taxis SET fare_per_distance = fare_amount / trip_distance WHERE trip_distance > 0")

spark.sql("SELECT trip_distance, fare_amount, fare_per_distance FROM demo.nyc.taxis LIMIT 5").show()

### Nâng cao: Time Travel & Snapshots
Kiểm tra lịch sử thay đổi của bảng thông qua metadata tables.

In [ ]:
# Xem các snapshot đã tạo
spark.sql("SELECT committed_at, snapshot_id, operation FROM demo.nyc.taxis.snapshots").show(truncate=False)

# Bạn có thể rollback hoặc query snapshot cũ (time travel)
# Ví dụ query: SELECT * FROM demo.nyc.taxis TIMESTAMP AS OF '2024-01-01 00:00:00'